# 📈 Project 2: Sales Forecasting & Time Series Analysis
**Domain:** Retail & E-Commerce Data Analytics  
**Objective:** Build an end-to-end forecasting pipeline to accurately predict future retail revenue by modeling long-term trends, weekly & annual seasonality, and promotional sales lifts.

---
### 🎯 Key Project Highlights:
1. **Stationarity & Statistical Testing:** Augmented Dickey-Fuller (ADF) tests and rolling variance assessment.
2. **Time Series Decomposition:** Separation into Trend, Multi-seasonal (7-day and 365-day), and Residual noise.
3. **Promotional Lift Analysis:** Quantifying demand spikes during promotional events, discount tiers, and weekend traffic.
4. **Model Comparison:** Seasonal Naive Baseline vs. Holt-Winters Exponential Smoothing vs. SARIMAX (with Exogenous Regressors) vs. Random Forest ML Benchmark.
5. **Holdout Evaluation:** 90-day test set validation using MAE, RMSE, MAPE, and $R^2$.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Data Loading & Exploratory Data Analysis (EDA)

In [2]:
df = pd.read_csv('data/retail_sales_daily.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Total Observations: {len(df)} days")
df.head()

## 2. Stationarity Testing (Augmented Dickey-Fuller)

In [3]:
adf_test = adfuller(df['sales'])
print('--- ADF Test Results ---')
print(f'ADF Statistic: {adf_test[0]:.4f}')
print(f'p-value: {adf_test[1]:.4f}')
print('Critical Values:', adf_test[4])
if adf_test[1] > 0.05:
    print('=> Non-Stationary: First-order differencing (d=1) required for ARIMA modeling.')
else:
    print('=> Series is Stationary.')

## 3. Time Series Decomposition (Trend, Seasonality, Residuals)

In [4]:
decomp = seasonal_decompose(df['sales'], model='additive', period=7)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.suptitle('Additive Decomposition (7-Day Seasonality)', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 4. Promotional Impact & Calendar Demand Lift

In [5]:
reg_mean = df[df['promotional_event'] == 0]['sales'].mean()
promo_mean = df[df['promotional_event'] == 1]['sales'].mean()
lift = ((promo_mean - reg_mean) / reg_mean) * 100
print(f'Regular Days Mean Sales:      ${reg_mean:,.2f}')
print(f'Promotional Days Mean Sales:  ${promo_mean:,.2f}')
print(f'Promotional Sales Lift:       +{lift:.1f}%')

## 5. Forecasting Models & Out-of-Sample 90-Day Holdout Evaluation

In [6]:
test_days = 90
train_df = df.iloc[:-test_days]
test_df = df.iloc[-test_days:]

# 1. Holt-Winters
hw = ExponentialSmoothing(train_df['sales'], trend='add', seasonal='add', seasonal_periods=7).fit()
hw_pred = hw.forecast(test_days)

# 2. SARIMAX with Exogenous Promos
exog_cols = ['promotional_event', 'discount_pct', 'is_weekend']
sarimax = SARIMAX(train_df['sales'], exog=train_df[exog_cols], order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
sarimax_pred = sarimax.get_forecast(steps=test_days, exog=test_df[exog_cols]).predicted_mean

# Load summary metrics
metrics_df = pd.read_csv('data/evaluation_metrics.csv')
metrics_df

## 6. Visualization: Forecast vs Actual Comparison

In [7]:
plt.figure(figsize=(15, 6))
plt.plot(test_df.index, test_df['sales'], label='Actual Sales', color='black', linewidth=2)
plt.plot(test_df.index, sarimax_pred.values, label='SARIMAX (with Exog Promos)', color='#2563eb', linewidth=2)
plt.plot(test_df.index, hw_pred.values, label='Holt-Winters', color='#10b981', linestyle='--')
plt.title('90-Day Sales Forecast vs Actual', fontsize=14, fontweight='bold')
plt.ylabel('Sales ($)')
plt.legend()
plt.show()